In [6]:
# ============================================================================
# DKTC 레이블 정제 v2.1 — label_name 형식 불일치 버그 수정
# ============================================================================
#
# v2.0 버그:
#   - 오분류 CSV: true_label = "협박", "갈취", ...
#   - 원본 CSV:  label_name = "협박 대화", "갈취 대화", ...
#   → 매핑 없이 비교/덮어쓰기 → train 변경 0건 + val 클래스 분열
#
# v2.1 수정:
#   - 원본 label_name을 자동 감지해서 매핑 테이블 생성
#   - 모든 비교/덮어쓰기에서 원본 형식을 유지
# ============================================================================

In [7]:
import os
import numpy as np
import pandas as pd

In [8]:
# ============================================================================
# 0. 레이블 형식 자동 감지 및 매핑
# ============================================================================
 
SHORT_LABELS = ["협박", "갈취", "직장내괴롭힘", "기타괴롭힘", "일반대화"]
 
def detect_label_format(df, label_name_col="label_name"):
    """
    원본 CSV의 label_name 형식을 감지하고, 짧은 형식 ↔ 원본 형식 매핑을 반환.
    """
    unique_labels = sorted(df[label_name_col].unique().tolist())
    
    if "협박" in unique_labels:
        short_to_orig = {s: s for s in SHORT_LABELS}
        orig_to_short = {s: s for s in SHORT_LABELS}
        print(f"  레이블 형식: 짧은 형식 (매핑 불필요)")
    else:
        short_to_orig = {}
        orig_to_short = {}
        
        for short_name in SHORT_LABELS:
            matched = None
            short_core = short_name.replace(" ", "")
            
            for orig_name in unique_labels:
                orig_core = orig_name.replace(" ", "").replace("대화", "")
                if orig_core == short_core:
                    matched = orig_name
                    break
            
            if matched is None:
                for orig_name in unique_labels:
                    if short_name[:2] in orig_name:
                        if orig_name not in orig_to_short:
                            matched = orig_name
                            break
            
            if matched:
                short_to_orig[short_name] = matched
                orig_to_short[matched] = short_name
            else:
                print(f"  [WARNING] '{short_name}'에 대한 원본 매핑을 찾지 못함")
                print(f"  원본 레이블 목록: {unique_labels}")
                short_to_orig[short_name] = short_name
                orig_to_short[short_name] = short_name
        
        print(f"  레이블 형식: 긴 형식")
        print(f"  매핑 테이블:")
        for short, orig in short_to_orig.items():
            print(f"    {short:12s} ↔ {orig}")
    
    return short_to_orig, orig_to_short

In [9]:
# ============================================================================
# PART 1: Val set 정제
# ============================================================================
 
def relabel_val_from_misclassified(val_df, misclassified_csv_path, margin_threshold=0.5):
    mis_df = pd.read_csv(misclassified_csv_path)
    
    print(f"\n  [Val 형식 감지]")
    short_to_orig, orig_to_short = detect_label_format(val_df)
    
    label_id_map = val_df.drop_duplicates(subset=["label", "label_name"])\
                         .set_index("label_name")["label"].to_dict()
    print(f"  label ID 매핑: {label_id_map}")
    
    THREAT_KW = [
        "죽여", "죽인다", "죽일", "죽는다", "죽어",
        "칼로", "찌른다", "찔러", "찌를",
        "불질러", "불지른다", "불태워",
        "납치", "폭파", "해코지",
        "목을", "목숨", "숨통", "패죽", "때려죽",
    ]
    EXTORTION_KW = [
        "돈", "만원", "천원", "백만",
        "빌려", "줘봐", "주라", "내놔", "내놓",
        "사줘", "사와", "사다줘",
        "가져와", "가져오",
        "입금", "송금", "이더리움", "비트코인",
        "이자", "원금",
    ]
    WORKPLACE_KW = [
        "대리", "과장", "부장", "차장", "팀장", "사원", "직원",
        "인턴", "신입", "법인카드", "회식", "야근", "출근",
        "부서", "본부", "팀에", "회의",
    ]
    
    candidates = mis_df[mis_df["margin"] >= margin_threshold].copy()
    print(f"\n  오분류 전체: {len(mis_df)}건")
    print(f"  margin >= {margin_threshold} 후보: {len(candidates)}건")
    
    val_fixed = val_df.copy()
    val_fixed["label_original"] = val_fixed["label"].copy()
    val_fixed["label_name_original"] = val_fixed["label_name"].copy()
    val_fixed["relabel_rule"] = ""
    
    changes = []
    
    for _, cand in candidates.iterrows():
        orig_idx = int(cand["orig_index"])
        text = str(cand["text"])
        true_label_short = cand["true_label"]
        pred_label_short = cand["pred_label"]
        margin = cand["margin"]
        
        has_threat = any(kw in text for kw in THREAT_KW)
        has_extortion = any(kw in text for kw in EXTORTION_KW)
        has_workplace = any(kw in text for kw in WORKPLACE_KW)
        
        new_label_short = None
        rule = ""
        
        if true_label_short == "협박" and pred_label_short == "기타괴롭힘" and not has_threat:
            new_label_short = "기타괴롭힘"
            rule = "Rule A: 협박→기타괴롭힘 (위해 고지 없음, 모델 동의)"
        elif true_label_short == "협박" and pred_label_short == "갈취" and has_extortion:
            new_label_short = "갈취"
            rule = "Rule B: 협박→갈취 (재물 요구 있음, 모델 동의)"
        elif true_label_short == "직장내괴롭힘" and pred_label_short == "기타괴롭힘" and not has_workplace:
            new_label_short = "기타괴롭힘"
            rule = "Rule C: 직장내괴롭힘→기타괴롭힘 (직장 맥락 없음, 모델 동의)"
        elif true_label_short == "기타괴롭힘" and pred_label_short == "협박" and has_threat:
            new_label_short = "협박"
            rule = "Rule D: 기타괴롭힘→협박 (위해 고지 있음, 모델 동의)"
        elif true_label_short == "기타괴롭힘" and pred_label_short == "갈취":
            ext_count = sum(1 for kw in EXTORTION_KW if kw in text)
            if ext_count >= 2:
                new_label_short = "갈취"
                rule = f"Rule E: 기타괴롭힘→갈취 (재물 키워드 {ext_count}개, 모델 동의)"
        
        if new_label_short is not None and new_label_short != true_label_short:
            new_label_orig = short_to_orig[new_label_short]
            new_label_id = label_id_map.get(new_label_orig)
            
            if new_label_id is None:
                print(f"  [WARNING] '{new_label_orig}'의 label ID를 찾지 못함, 건너뜀")
                continue
            
            if orig_idx in val_fixed.index:
                val_fixed.at[orig_idx, "label_name"] = new_label_orig
                val_fixed.at[orig_idx, "label"] = new_label_id
                val_fixed.at[orig_idx, "relabel_rule"] = rule
            
            changes.append({
                "orig_index": orig_idx,
                "text_preview": text[:100],
                "original_label": true_label_short,
                "model_prediction": pred_label_short,
                "new_label_short": new_label_short,
                "new_label_orig": new_label_orig,
                "margin": margin,
                "rule": rule,
            })
    
    return val_fixed, pd.DataFrame(changes)

In [10]:
# ============================================================================
# PART 2: Train set 정제
# ============================================================================
 
def relabel_train_conservative(train_df, val_change_log):
    print(f"\n  [Train 형식 감지]")
    short_to_orig, orig_to_short = detect_label_format(train_df)
    
    label_id_map = train_df.drop_duplicates(subset=["label", "label_name"])\
                           .set_index("label_name")["label"].to_dict()
    print(f"  label ID 매핑: {label_id_map}")
    
    THREAT_KW = [
        "죽여", "죽인다", "죽일", "죽는다", "죽어",
        "칼로", "찌른다", "찔러", "찌를",
        "불질러", "불지른다", "불태워",
        "납치", "폭파", "해코지",
        "목을", "목숨", "숨통", "패죽", "때려죽",
    ]
    EXTORTION_KW_STRICT = [
        "만원", "천원", "백만",
        "내놔", "내놓",
        "사줘", "사다줘",
        "가져와", "가져오",
        "입금", "송금", "이더리움",
    ]
    WORKPLACE_KW = [
        "대리", "과장", "부장", "차장", "팀장", "사원", "직원",
        "인턴", "신입", "법인카드", "회식", "야근", "출근",
        "부서", "본부", "팀에", "회의",
    ]
    
    active_rules = set()
    if len(val_change_log) > 0:
        for rule in val_change_log["rule"].unique():
            rule_id = rule.split(":")[0].strip()
            active_rules.add(rule_id)
    print(f"  val에서 활성화된 규칙: {active_rules}")
    
    df = train_df.copy()
    df["label_original"] = df["label"].copy()
    df["label_name_original"] = df["label_name"].copy()
    df["relabel_rule"] = ""
    
    changes = []
    
    for idx, row in df.iterrows():
        text = str(row["text"])
        current_label_orig = row["label_name"]
        
        current_label_short = orig_to_short.get(current_label_orig)
        if current_label_short is None:
            continue
        
        new_label_short = current_label_short
        rule = ""
        
        has_threat = any(kw in text for kw in THREAT_KW)
        has_extortion_strict = any(kw in text for kw in EXTORTION_KW_STRICT)
        has_workplace = any(kw in text for kw in WORKPLACE_KW)
        
        if "Rule A" in active_rules:
            if current_label_short == "협박" and not has_threat and not has_extortion_strict:
                new_label_short = "기타괴롭힘"
                rule = "Rule A: 협박→기타괴롭힘 (위해 고지 없음)"
        
        if "Rule B" in active_rules and new_label_short == current_label_short:
            if current_label_short == "협박" and has_extortion_strict:
                new_label_short = "갈취"
                rule = "Rule B: 협박→갈취 (재물 요구 있음)"
        
        if "Rule C" in active_rules and new_label_short == current_label_short:
            if current_label_short == "직장내괴롭힘" and not has_workplace:
                new_label_short = "기타괴롭힘"
                rule = "Rule C: 직장내괴롭힘→기타괴롭힘 (직장 맥락 없음)"
        
        if "Rule D" in active_rules and new_label_short == current_label_short:
            if current_label_short == "기타괴롭힘" and has_threat and not has_extortion_strict:
                new_label_short = "협박"
                rule = "Rule D: 기타괴롭힘→협박 (위해 고지 있음)"
        
        if new_label_short != current_label_short:
            new_label_orig = short_to_orig[new_label_short]
            new_label_id = label_id_map.get(new_label_orig)
            
            if new_label_id is None:
                print(f"  [WARNING] '{new_label_orig}'의 label ID를 찾지 못함, 건너뜀")
                continue
            
            df.at[idx, "label_name"] = new_label_orig
            df.at[idx, "label"] = new_label_id
            df.at[idx, "relabel_rule"] = rule
            changes.append({
                "index": idx,
                "text_preview": text[:100],
                "original_label": current_label_orig,
                "new_label": new_label_orig,
                "rule": rule,
            })
    
    return df, pd.DataFrame(changes)

In [11]:
# ============================================================================
# PART 3: 실행
# ============================================================================
if __name__ == "__main__":
    
    print("=" * 70)
    print("DKTC 레이블 정제 v2.1 — label_name 형식 불일치 수정")
    print("=" * 70)
    
    TRAIN_CSV = "data/train_processed_260317_n_1000.csv"
    VAL_CSV = "data/val_processed_260317_n_1000.csv"
    MISCLASSIFIED_CSV = "dktc_all_misclassified_with_top2.csv"
    MARGIN_THRESHOLD = 0.5
    
    train_df = pd.read_csv(TRAIN_CSV)
    val_df = pd.read_csv(VAL_CSV)
    
    print(f"\n[원본] Train: {len(train_df)}, Val: {len(val_df)}")
    print(f"\n원본 Train 클래스 분포:")
    print(train_df["label_name"].value_counts().to_string())
    print(f"\n원본 Val 클래스 분포:")
    print(val_df["label_name"].value_counts().to_string())
    
    # Step 1
    print(f"\n\n{'=' * 70}")
    print(f"Step 1: Val set 정제 (margin >= {MARGIN_THRESHOLD})")
    print(f"{'=' * 70}")
    
    if not os.path.exists(MISCLASSIFIED_CSV):
        print(f"\n[ERROR] {MISCLASSIFIED_CSV} 파일이 없습니다.")
        exit(1)
    
    val_fixed, val_changes = relabel_val_from_misclassified(
        val_df, MISCLASSIFIED_CSV, margin_threshold=MARGIN_THRESHOLD
    )
    
    print(f"\n  Val 변경: {len(val_changes)}건")
    if len(val_changes) > 0:
        print(f"\n  규칙별 건수:")
        print(val_changes["rule"].value_counts().to_string())
    
    # Step 2
    print(f"\n\n{'=' * 70}")
    print(f"Step 2: Train set 정제 (val에서 검증된 규칙만, 보수적 적용)")
    print(f"{'=' * 70}")
    
    train_fixed, train_changes = relabel_train_conservative(train_df, val_changes)
    
    print(f"\n  Train 변경: {len(train_changes)}건 / {len(train_df)}건 "
          f"({len(train_changes)/len(train_df)*100:.1f}%)")
    if len(train_changes) > 0:
        print(f"\n  규칙별 건수:")
        print(train_changes["rule"].value_counts().to_string())
    
    # 분포 비교
    print(f"\n\n{'=' * 70}")
    print("변경 전후 클래스 분포 비교")
    print(f"{'=' * 70}")
    
    for name, orig, fixed in [("Train", train_df, train_fixed), ("Val", val_df, val_fixed)]:
        print(f"\n[{name}]")
        orig_dist = orig["label_name"].value_counts()
        fixed_dist = fixed["label_name"].value_counts()
        comp = pd.DataFrame({"원본": orig_dist, "수정후": fixed_dist}).fillna(0).astype(int)
        comp["변화"] = comp["수정후"] - comp["원본"]
        print(comp.to_string())
    
    # Sanity Check
    print(f"\n\n{'=' * 70}")
    print("Sanity Check")
    print(f"{'=' * 70}")
    
    orig_labels_v = set(val_df["label_name"].unique())
    fixed_labels_v = set(val_fixed["label_name"].unique())
    new_labels_v = fixed_labels_v - orig_labels_v
    if new_labels_v:
        print(f"  [FAIL] Val에 새로운 label_name 등장: {new_labels_v}")
    else:
        print(f"  [PASS] Val label_name 형식 일관성 유지")
    
    orig_labels_t = set(train_df["label_name"].unique())
    fixed_labels_t = set(train_fixed["label_name"].unique())
    new_labels_t = fixed_labels_t - orig_labels_t
    if new_labels_t:
        print(f"  [FAIL] Train에 새로운 label_name 등장: {new_labels_t}")
    else:
        print(f"  [PASS] Train label_name 형식 일관성 유지")
    
    assert len(val_fixed) == len(val_df), "Val 행 수 변경됨!"
    assert len(train_fixed) == len(train_df), "Train 행 수 변경됨!"
    print(f"  [PASS] 행 수 불변: Train {len(train_fixed)}, Val {len(val_fixed)}")
    
    # 저장
    train_fixed.to_csv("data/train_relabeled_v2.csv", index=False, encoding="utf-8-sig")
    val_fixed.to_csv("data/val_relabeled_v2.csv", index=False, encoding="utf-8-sig")
    train_changes.to_csv("data/train_relabel_v2_changelog.csv", index=False, encoding="utf-8-sig")
    val_changes.to_csv("data/val_relabel_v2_changelog.csv", index=False, encoding="utf-8-sig")
    
    print(f"\n\n저장 완료:")
    print(f"  data/train_relabeled_v2.csv")
    print(f"  data/val_relabeled_v2.csv")
    print(f"  data/train_relabel_v2_changelog.csv")
    print(f"  data/val_relabel_v2_changelog.csv")
    print(f"\n→ HP 튜닝 노트북 셀 01에서 경로 변경:")
    print(f'  train_df = pd.read_csv("data/train_relabeled_v2.csv")')
    print(f'  val_df   = pd.read_csv("data/val_relabeled_v2.csv")')

DKTC 레이블 정제 v2.1 — label_name 형식 불일치 수정

[원본] Train: 3942, Val: 969

원본 Train 클래스 분포:
label_name
기타 괴롭힘 대화      808
일반 대화          800
협박 대화          778
갈취 대화          778
직장 내 괴롭힘 대화    778

원본 Val 클래스 분포:
label_name
기타 괴롭힘 대화      202
일반 대화          200
갈취 대화          195
직장 내 괴롭힘 대화    194
협박 대화          178


Step 1: Val set 정제 (margin >= 0.5)

  [Val 형식 감지]
  레이블 형식: 긴 형식
  매핑 테이블:
    협박           ↔ 협박 대화
    갈취           ↔ 갈취 대화
    직장내괴롭힘       ↔ 직장 내 괴롭힘 대화
    기타괴롭힘        ↔ 기타 괴롭힘 대화
    일반대화         ↔ 일반 대화
  label ID 매핑: {'기타 괴롭힘 대화': 3, '갈취 대화': 1, '직장 내 괴롭힘 대화': 2, '협박 대화': 0, '일반 대화': 4}

  오분류 전체: 71건
  margin >= 0.5 후보: 60건

  Val 변경: 27건

  규칙별 건수:
rule
Rule A: 협박→기타괴롭힘 (위해 고지 없음, 모델 동의)        11
Rule B: 협박→갈취 (재물 요구 있음, 모델 동의)           10
Rule C: 직장내괴롭힘→기타괴롭힘 (직장 맥락 없음, 모델 동의)     3
Rule E: 기타괴롭힘→갈취 (재물 키워드 3개, 모델 동의)        1
Rule E: 기타괴롭힘→갈취 (재물 키워드 2개, 모델 동의)        1
Rule D: 기타괴롭힘→협박 (위해 고지 있음, 모델 동의)         1


Step 2: Train set 정제 (val에서 검증된 규칙만, 보수적 적용)

